# Stochastic SINDy (SSISDE): Black–Scholes experiment

Run all cells from a fresh Python 3.10–3.12 kernel (tested locally with 3.11).
The original scientific code is preserved in `data_generation/`, `models/`, and
the experiment cells below. `%run -i` runs each script in this notebook's shared
namespace, maintaining the original cell order and global variables.

Local VS Code: select `.venv/bin/python`. Colab in VS Code: select **Colab → Auto
Connect**. A remote runtime needs its own copy of the scripts; the setup cell
downloads the private repository when it is not already present, asking for a
GitHub token through a hidden input. Use a fine-grained token with read-only
Contents access to `munawarali93/SSISDE`. Do not paste a token into a code cell.
An existing runtime checkout is reused; restart/delete it to fetch newer code.

Figures keep their original filenames and are saved in `experiments/`.
The untouched input notebook is in `original/`. See `README.md` for setup details.


In [ ]:
from pathlib import Path
import os
import sys

if not (3, 10) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("The original dependency pins require Python 3.10–3.12; use the local .venv or a compatible Colab runtime.")

def _find_project():
    for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/SSISDE')]:
        if (candidate / 'models/stochastic_sindy.py').is_file() and (candidate / 'requirements.txt').is_file():
            return candidate
    return None

PROJECT_ROOT = _find_project()
if PROJECT_ROOT is None:
    import importlib.util
    try:
        _in_colab = importlib.util.find_spec('google.colab') is not None
    except ModuleNotFoundError:
        _in_colab = False
    if not _in_colab:
        raise FileNotFoundError("Open this notebook from the SSISDE checkout so its Python scripts are available.")

    def _download_private_project():
        from getpass import getpass
        from io import BytesIO
        from urllib.request import Request, urlopen
        from zipfile import ZipFile
        token = getpass('GitHub token (read-only Contents access to SSISDE): ')
        try:
            request = Request(
                'https://api.github.com/repos/munawarali93/SSISDE/zipball/main',
                headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json'},
            )
            with urlopen(request, timeout=120) as response:
                archive_bytes = response.read()
        finally:
            token = None
            request = None
        destination = Path('/content/SSISDE')
        with ZipFile(BytesIO(archive_bytes)) as archive:
            for member in archive.infolist():
                relative = Path(*Path(member.filename).parts[1:])
                if member.is_dir() or not relative.parts:
                    continue
                target = (destination / relative).resolve()
                if not target.is_relative_to(destination.resolve()):
                    raise ValueError('Unsafe archive path')
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(archive.read(member))
        return destination

    PROJECT_ROOT = _download_private_project()

os.chdir(PROJECT_ROOT / 'experiments')
print(f'Project: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')


##Install libraries

In [ ]:
# Install into the selected kernel before importing scientific libraries.
# The four original version pins are retained in requirements.txt.
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')])


##Import libraries

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
import pysindy as ps
import random
import iisignature
from sklearn import linear_model
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.model_selection import TimeSeriesSplit

##Defining parameters and simulating noise

In [ ]:
%run -i ../data_generation/parameters.py


##Simulating primary processes

## $$dS_t = \mu(S_t)dt + \sigma(S_t)dB_t$$

In [ ]:
%run -i ../data_generation/processes.py
plt.plot(S_, label="S")
plt.legend()
plt.show()

##compute quadratic variation

In [ ]:
%run -i ../models/quadratic_variation.py
plt.plot(QV)

##Approximating $\sigma$

In [ ]:
%run -i ../models/diffusion.py


In [ ]:
sigma_pred_ = np.sqrt(model.predict(S_.reshape(-1, 1)))
sigma_pred_ = np.array(sigma_pred_)
plt.plot(sigma_exact_, label="Actual sigma",color="blue")
plt.plot(sigma_pred_,label="predicted sigma", color="orange")
plt.legend()
plt.show()

In [ ]:
%run -i ../models/diffusion_derivatives.py


In [ ]:
plt.plot(sigma_pred_)

##Q Brownian motion

In [ ]:
%run -i ../models/q_brownian_motion.py
plt.plot(BQ_)

#Fit signature model

In [ ]:
keep_cols = [0, 1, 3, 4, 5, 13]

In [ ]:
%run -i ../models/signature_regression.py
plt.plot(l21)
np.mean(l21)

In [ ]:
%run -i ../data_generation/coarse_graining.py


In [ ]:
%run -i ../models/coarse_diffusion_derivatives.py


In [ ]:
plt.plot(sigma_2prime)

## Predicting $\mu(X_t)$ by tuncating up to level-2 of iterated integrals

In [ ]:
%run -i ../models/drift_coefficients.py


In [ ]:
#plt.plot(b)
plt.plot(a)

In [ ]:
%run -i ../models/drift_recovery.py


In [ ]:
t_ = np.linspace(0, T, K)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(t_, mu_pred1, label="Recovered drift")
axes[0].plot(t_, mu_exact, label="True drift")
axes[0].set_title("Drift via Method 1")
axes[0].set_xlabel("t")
axes[0].set_ylabel("$\mu(X_t)$")
axes[0].legend()
axes[1].plot(t_, mu_pred2, label="Recovered drift")
axes[1].set_title("Drift via Method 2")
axes[1].plot(t_, mu_exact, '--', label="True drift")
axes[1].set_xlabel("t")
axes[1].set_ylabel("$\mu(X_t)$")
axes[1].legend()
axes[1].set_ylim(axes[0].get_ylim())

plt.tight_layout()
plt.savefig("Drift_via_Euler_BS.png", bbox_inches='tight', dpi=300)
plt.show()

##Predicting P-Brownian motion

In [ ]:
%run -i ../models/p_brownian_motion.py
plt.plot(BP, label="Predicted Noise", linewidth=1)
plt.plot(W, label="Actual Noise", linewidth=1)
plt.plot(BQ, label="Q Brownian motion", linewidth=1)
plt.title("Actual and predicted noise with Q-BM")
plt.legend()
plt.savefig(f"{model}_Actual and predicted noise with Q-BM.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
%run -i ../models/stochastic_sindy.py
identifier = StochSINDy(
    mu_deg=5, si_deg=5,
    pen="elasticnet",
    use_cv=True, k=7,
    alphas=np.geomspace(1e-5, 1e1, 35),
    rhos=[0.3, 0.4, 0.5, 0.7, 0.85, 0.95, 1.0],
    #rhos=[0.7, 0.85, 0.95, 1.0],
    use_1se=True
)
identifier.fit(S, BP, dt)
identifier.show()
identifier.plot_cv()
identifier.plot_size(count_bias=True)
identifier.plot_results(S, BP, BQ, W, mu_true=mu_exact, si_true=sigma_exact)